In [0]:
from pyspark.sql.functions import *

---

# **Incremental Data Loading**
Incrementally load the data using auto loader using the Directory Listing method. Both Idempotency and Schema evolution are managed by:

- **File Loading Idempotency**: The checkpoint is stored in the `CheckpointLocation` that further has a `rocksdb`, this guarantees Idempotency and ensures only new files are processed everytime.

- **Schema Handling**: When the data is first loaded, the schema is inferred, cached and saved in `SchemaLocation` and if the schema evolved, it will first write the schema in the `SchemaLocation` and then try to match the cached schema and evolve the schema while writing the data in the location

In [0]:
# Get the source and target folder
dbutils.widgets.text("source_folder", "")
dbutils.widgets.text("target_folder", "")

# Get the source and target folder parameters values from above and store in the variable
src_folder = dbutils.widgets.get("source_folder")
tgt_folder = dbutils.widgets.get("target_folder")

base = "abfss://bronze@netflixprojectetedlll.dfs.core.windows.net"
checkpoint_path = f"{base}/checkpoints/{tgt_folder}"
schema_path     = f"{base}/schemas/{tgt_folder}"

In [0]:
# Incrementally Read and load the data from the raw container
df = spark.readStream.format("cloudFiles") \
                     .option("cloudFiles.format", "csv") \
                     .option("cloudFiles.schemaLocation", schema_path) \
                     .option("quote", '"') \
                     .option("escape", '"') \
                     .load(f"abfss://raw@netflixprojectetedlll.dfs.core.windows.net/{src_folder}")

In [0]:
# Add the ingestion date column for the incremental read in Silver
if tgt_folder == "netflix_titles":
    df = df.withColumn("_ingested_date", current_timestamp()) \
           .withColumn("_file_modified", col("_metadata.file_modification_time"))

In [0]:
# Write the data in Bronze layer
df.writeStream.format("delta") \
              .outputMode("append") \
              .option("checkpointLocation", checkpoint_path) \
              .option("mergeSchema", "true") \
              .trigger(availableNow=True) \
              .start(f"abfss://bronze@netflixprojectetedlll.dfs.core.windows.net/{tgt_folder}")

---